# PPO Explained — CarRacing Self-Driving Project

This notebook explains **Proximal Policy Optimization (PPO)** in interview-friendly language and maps each idea to this repo's Stable-Baselines3 training pipeline on Gymnasium **CarRacing-v3** (maintained equivalent of Gym `CarRacing-v0`).

We do **not** re-implement PPO here — training lives in `src/train.py`. The goal is to understand *why* the clipped surrogate, GAE, and frame stacking matter.

## 1. MDP setup for CarRacing

| Component | In this project |
|-----------|-----------------|
| **State** $s_t$ | Pixel observations: grayscale 84×84 frames, stacked ×4 → shape `(4, 84, 84)` |
| **Action** $a_t$ | Continuous: `[steer, gas, brake]` ∈ Box |
| **Reward** $r_t$ | Dense: progress along the track minus a small time penalty; leaving the track hurts return |
| **Dynamics** | Box2D physics; partially observed through pixels |

Goal: learn a policy $\pi_\theta(a\mid s)$ that maximizes expected discounted return
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\Big[\sum_{t=0}^{T} \gamma^t r_t\Big].$$

## 2. Actor–critic

PPO is an **actor–critic** method:

- **Actor (policy)** $\pi_\theta(a\mid s)$ — chooses steering / gas / brake. In SB3 we use `CnnPolicy`: a CNN feature extractor + Gaussian head for continuous actions.
- **Critic (value)** $V_\phi(s)$ — estimates expected return from state $s$. Used to reduce variance of policy updates.

In code: `PPO(..., policy="CnnPolicy")` builds both networks sharing a CNN backbone (see SB3's `NatureCNN`).

## 3. Advantage estimation (GAE)

The **advantage** $A_t$ answers: *was this action better or worse than average for this state?*

$$A_t = Q(s_t, a_t) - V(s_t).$$

**Generalized Advantage Estimation (GAE)** blends multi-step TD residuals:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t),$$
$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}.$$

- $\lambda \to 0$: low variance, more bias (1-step TD)
- $\lambda \to 1$: Monte-Carlo-like, higher variance

Config defaults: `gamma=0.99`, `gae_lambda=0.95` in `configs/ppo_carracing.yaml`.

## 4. Clipped surrogate objective (why PPO is stable)

Vanilla policy gradient can take **destructive** steps when the likelihood ratio
$$r_t(\theta) = \frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\text{old}}}(a_t\mid s_t)}$$
gets large.

PPO's clipped objective:
$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[\min\big(r_t(\theta)\hat{A}_t,\;\mathrm{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\big)\Big].$$

With $\epsilon = 0.2$ (`clip_range`), updates that push the new policy too far from the old one are **ignored** (the min picks the pessimistic bound). That is the "proximal" in PPO.

Total SB3 loss also includes value loss and entropy bonus (`vf_coef`, `ent_coef`) for exploration.

## 5. How SB3 maps theory → code

| Theory | SB3 / this repo |
|--------|------------------|
| Collect on-policy rollouts | `n_steps` per env → `RolloutBuffer` |
| GAE advantages | computed inside buffer using `gamma`, `gae_lambda` |
| Clipped policy update | `clip_range`; multiple `n_epochs` over minibatches of size `batch_size` |
| CNN features from pixels | `CnnPolicy` |
| Parallel data collection | `SubprocVecEnv` / `DummyVecEnv` in `src/env.py` |
| Best checkpoint | `EvalCallback` → `models/best/best_model.zip` |
| Curves | TensorBoard under `logs/tb/` |

Entry point: `python -m src.train --config configs/ppo_carracing.yaml`.

## 6. Why preprocessing & frame stacking matter

A single RGB frame is **not Markov** for velocity: you cannot tell speed or turn rate from one still image.

Pipeline in `src/env.py`:
1. **Grayscale** — cuts channels, keeps track geometry
2. **Resize 84×84** — standard Atari-style CNN input
3. **Frame stack (4)** — approximate velocity / angular rate from differences across frames
4. **Channel-first `(4,84,84)`** — what `CnnPolicy` expects

Without stacking, the policy must invent dynamics from one snapshot — usually fails to stay on track.

## 7. Interview talking points (60–90 seconds)

1. **Problem**: learn continuous control from pixels in CarRacing (progress reward).
2. **Algo**: PPO actor–critic — policy + value CNN; on-policy rollouts.
3. **Stability**: clipped surrogate prevents huge policy ratio jumps; GAE balances bias/variance.
4. **Engineering**: grayscale + 84×84 + 4-frame stack; VecEnv; EvalCallback for best model; TensorBoard.
5. **Env note**: Gymnasium `CarRacing-v3` ≈ legacy Gym `CarRacing-v0`.

See the project `README.md` for train / eval / demo commands and results.

In [ ]:
# Optional: peek at config hyperparameters used for training
from pathlib import Path
import yaml

cfg_path = Path("../configs/ppo_carracing.yaml")
if not cfg_path.exists():
    cfg_path = Path("configs/ppo_carracing.yaml")

with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

for k in ["env_id", "learning_rate", "n_steps", "batch_size", "n_epochs",
          "gamma", "gae_lambda", "clip_range", "ent_coef", "frame_stack"]:
    print(f"{k:16s} {cfg.get(k)}")